<a href="https://colab.research.google.com/github/nitijain18/style-buddy/blob/main/model2_sb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opendatasets

In [2]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small?utm_source=chatgpt.com")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: nitijain13
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small


100%|██████████| 565M/565M [00:33<00:00, 17.6MB/s]


In [3]:
import os
import pandas as pd

image_dir = "/content/fashion-product-images-small/images"
data = pd.read_csv("/content/fashion-product-images-small/styles.csv" , on_bad_lines="skip")


In [4]:
data.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [5]:
selected_articles = [
    "Tshirts", "Shirts", "Tops", "Kurtas",
    "Jeans", "Trousers", "Shorts",
    "Dresses",
    "Handbags", "Backpacks",
    "Flats", "Formal Shoes", "Heels"
]

sub_data = data[data["articleType"].isin(selected_articles)].copy()

sub_data["image_path"] = sub_data["id"].astype(str).apply(
    lambda x: os.path.join(image_dir, x + ".jpg")
)

sub_data = sub_data[sub_data["image_path"].apply(os.path.exists)]

print(sub_data["articleType"].value_counts())
print("Total images:", len(sub_data))

articleType
Tshirts         7066
Shirts          3215
Kurtas          1844
Tops            1762
Handbags        1759
Heels           1323
Backpacks        724
Formal Shoes     637
Jeans            608
Shorts           547
Trousers         530
Flats            500
Dresses          464
Name: count, dtype: int64
Total images: 20979


In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    sub_data,
    test_size=0.2,
    stratify=sub_data["articleType"],
    random_state=42
)

In [7]:
print(len(train_df))
print(len(test_df))

16783
4196


In [8]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224,224)
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_gen = ImageDataGenerator(
    rescale=1./255
)

train_ds = train_gen.flow_from_dataframe(
    train_df,
    x_col="image_path",
    y_col="articleType",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_ds = test_gen.flow_from_dataframe(
    test_df,
    x_col="image_path",
    y_col="articleType",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 16783 validated image filenames belonging to 13 classes.
Found 4196 validated image filenames belonging to 13 classes.


In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Dropout

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(13, activation="softmax")
])

model.compile(
    optimizer="Adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 2048)           │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 13)             │         3,341 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,115,597 (91.99 MB)

 Trainable params: 527,885 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [13]:
# fine-tuning ResNet50
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)

Epoch 1/5
525/525 ━━━━━━━━━━━━━━━━━━━━ 262s 460ms/step - accuracy: 0.6137 - loss: 1.1194 - val_accuracy: 0.7378 - val_loss: 0.7439
Epoch 2/5
525/525 ━━━━━━━━━━━━━━━━━━━━ 227s 432ms/step - accuracy: 0.7119 - loss: 0.7774 - val_accuracy: 0.7888 - val_loss: 0.5800
Epoch 3/5
525/525 ━━━━━━━━━━━━━━━━━━━━ 230s 438ms/step - accuracy: 0.7408 - loss: 0.7027 - val_accuracy: 0.8048 - val_loss: 0.5337
Epoch 4/5
525/525 ━━━━━━━━━━━━━━━━━━━━ 226s 431ms/step - accuracy: 0.7564 - loss: 0.6554 - val_accuracy: 0.7612 - val_loss: 0.6458
Epoch 5/5
525/525 ━━━━━━━━━━━━━━━━━━━━ 228s 435ms/step - accuracy: 0.7684 - loss: 0.6273 - val_accuracy: 0.7786 - val_loss: 0.5939


In [14]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test Accuracy:", test_acc)

132/132 ━━━━━━━━━━━━━━━━━━━━ 13s 96ms/step - accuracy: 0.7786 - loss: 0.5939
Test Accuracy: 0.7785986661911011


In [15]:
import numpy as np
from sklearn.metrics import classification_report

pred = model.predict(test_ds)
y_pred = np.argmax(pred, axis=1)
y_true = test_ds.classes

print(classification_report(
    y_true,
    y_pred,
    target_names=list(test_ds.class_indices.keys())
))

132/132 ━━━━━━━━━━━━━━━━━━━━ 21s 123ms/step
              precision    recall  f1-score   support

   Backpacks       0.64      0.92      0.75       145
     Dresses       0.60      0.06      0.12        93
       Flats       1.00      0.01      0.02       100
Formal Shoes       0.98      0.97      0.97       127
    Handbags       0.89      0.88      0.89       352
       Heels       0.74      0.97      0.84       265
       Jeans       0.71      0.87      0.78       122
      Kurtas       0.83      0.88      0.85       369
      Shirts       0.63      0.96      0.76       643
      Shorts       0.80      0.76      0.78       109
        Tops       0.59      0.38      0.46       352
    Trousers       0.88      0.55      0.67       106
     Tshirts       0.91      0.79      0.84      1413

    accuracy                           0.78      4196
   macro avg       0.78      0.69      0.67      4196
weighted avg       0.80      0.78      0.76      4196



In [16]:
model.save("style_buddy_category_classifier-model2.keras")